# AI-Based Industrial Machine Health & Failure Prediction
## Machine Learning Capstone Project (23CSE301) — Review 1 Submission
**Academic Year:** 2026–2027 | **Review Scope:** Review 1 Phase Only (Total: 25 Marks)  
**Evaluator Note:** All cells are written in simple, modular, evaluator-friendly Python with clear explanations and zero hidden state.


## 1. Problem Statement
Modern industrial manufacturing relies on computerized numerical control (CNC) machines, milling equipment, and automated rotating assembly units. Unexpected mechanical or electrical machine breakdown causes expensive production downtime, damaged workpieces, and occupational safety hazards.

Industrial machinery experiences operating degradation driven by physical factors including thermal friction (`Air` and `Process Temperatures`), mechanical stress (`Rotational speed [rpm]` and `Torque [Nm]`), and cutting tool fatigue (`Tool wear [min]`).

The objective of this project is to build an end-to-end Machine Learning pipeline for predictive maintenance using the **AI4I 2020 Predictive Maintenance Dataset**:
1. **Regression Track**: Predict continuous tool wear accumulation (`Tool wear [min]`) to monitor equipment aging and forecast tool replacement schedules.
2. **Classification Track (Part A)**: Predict machine operational failure (`Machine failure`, binary 0/1) using baseline statistical and machine learning classifiers to trigger automated protective shutdowns.


## 2. Dataset Description
The dataset contains **10,000 operational records** with 14 variables representing synthetic but physically realistic milling machine operations:
- `UDI`: Row identifier (1 to 10,000) [Excluded from model features to prevent arbitrary index memorization].
- `Product ID`: Variant identifier consisting of quality letter (L, M, H) and serial number [Excluded from model features].
- `Type`: Machine product quality variant:
  - **L (Low)**: 50% of units (product variants with standard tolerances)
  - **M (Medium)**: 30% of units
  - **H (High)**: 20% of units (heavy-duty precision variants)
- `Air temperature [K]`: Generated using a random walk normalized to standard deviations around ~300 K.
- `Process temperature [K]`: Generated as air temperature + ~10 K thermal offset due to machine friction.
- `Rotational speed [rpm]`: Spindle speed calculated around an operating power band.
- `Torque [Nm]`: Resistance torque normally distributed around ~40 Nm.
- `Tool wear [min]`: Cumulative active cutting time before replacement.
- `Machine failure`: Ground-truth binary target (0 = Normal Operation, 1 = Machine Failure).
- `TWF`, `HDF`, `PWF`, `OSF`, `RNF`: Five specific failure modes (Tool Wear Failure, Heat Dissipation Failure, Power Failure, Overstrain Failure, Random Network Failure). *Audited below for target leakage.*


## 3. Import Libraries
We import standard scientific and machine learning libraries. All seeds are fixed to `42` for exact reproducibility.


In [ ]:
# Core scientific packages
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing & Model Selection
from sklearn.model_selection import train_test_split, KFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA

# 10 Regression Algorithms
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import r2_score, root_mean_squared_error, mean_absolute_error

# 5 Classification Algorithms (Part A)
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

# Configure plot styling
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['font.size'] = 11

print("All required libraries successfully loaded!")


## 4. Load Dataset
We load `ai4i2020.csv` directly from the local data directory and preview initial observations.


In [ ]:
# Load dataset
data_path = 'ai4i2020.csv' if os.path.exists('ai4i2020.csv') else 'data/ai4i2020.csv'
df = pd.read_csv(data_path)

print("Dataset successfully loaded.")
print(f"Dataset Dimensions: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()


## 5. Dataset Audit (Review 1 Rubric A1)
A comprehensive audit of dataset shape, data types, missing-value counts, duplicates, and target distributions.


In [ ]:
# Dataset Audit Report
print("=" * 60)
print("              DATASET AUDIT REPORT (Review 1 A1)            ")
print("=" * 60)
print(f"Dataset Shape: Rows = {df.shape[0]}, Columns = {df.shape[1]}")
print(f"Total Duplicate Rows: {df.duplicated().sum()}")
print("
--- Missing Value Counts per Column ---")
print(df.isnull().sum())

print("
--- Data Types & Non-Null Summary ---")
print(df.dtypes)

print("
--- Target Variable Distribution: Machine failure ---")
failure_counts = df['Machine failure'].value_counts()
failure_percentages = df['Machine failure'].value_counts(normalize=True) * 100
audit_target_df = pd.DataFrame({
    'Count': failure_counts,
    'Percentage (%)': failure_percentages.round(2)
})
print(audit_target_df)

print("
--- 5-Point Summary for Numerical Sensor Features ---")
sensor_cols = ['Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]']
print(df[sensor_cols].describe().T[['mean', 'std', 'min', '25%', '50%', '75%', 'max']].round(2))


### Dataset Audit Commentary
- **Shape & Completeness**: The dataset contains exactly 10,000 instances and 14 features with **0 missing values** and **0 duplicate entries**.
- **Data Types**: Contains 2 string columns (`Product ID`, `Type`), 3 float columns (`Air temperature`, `Process temperature`, `Torque`), and 9 integer columns (`UDI`, `Rotational speed`, `Tool wear`, `Machine failure`, and 5 failure flags).
- **Target Distribution**: Exactly 9,661 instances (96.61%) are non-failure cycles and 339 instances (3.39%) are machine failures. This confirms significant class imbalance (~28.5:1 ratio), which necessitates stratified splitting and threshold-independent metrics.


## 6. Exploratory Data Analysis (Review 1 Rubrics A2 & A3)
We conduct comprehensive EDA with 9 focused graphs designed to address specific engineering and machine learning questions. Every graph adheres strictly to the required format: **Purpose**, **Observation**, **Meaning**, and **ML Relevance**.


In [ ]:
# 6.1 Target Distribution Plot
fig, ax = plt.subplots(figsize=(7, 5))
counts = df['Machine failure'].value_counts()
bars = ax.bar(['No Failure (0)', 'Failure (1)'], counts.values, color=['#2b5c8f', '#d95f02'], width=0.5, edgecolor='black', linewidth=1.2)
for bar in bars:
    yval = bar.get_height()
    pct = (yval / len(df)) * 100
    ax.text(bar.get_x() + bar.get_width()/2.0, yval + 100, f'{yval:,} ({pct:.2f}%)', ha='center', va='bottom', fontweight='bold')
ax.set_title('Target Distribution: Machine Failure (ai4i2020)', fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Machine Condition Status', fontsize=11, fontweight='bold')
ax.set_ylabel('Observation Count', fontsize=11, fontweight='bold')
ax.set_ylim(0, 11000)
plt.tight_layout()
plt.show()


### Graph 1 — Target Distribution (Machine Failure)
- **Purpose**: Verify the balance between operating failure and normal running states in the production dataset.
- **Observation**: 9,661 machines (96.61%) did not fail, while 339 machines (3.39%) suffered operational failure.
- **Meaning**: Machine breakdowns are rare events under controlled operating conditions, reflecting realistic factory environments.
- **ML Relevance**: Standard classification accuracy is an uninformative metric (a trivial model predicting all 0s yields 96.61% accuracy). We must utilize **Stratified splitting**, **Precision**, **Recall**, **Weighted F1-score**, and **ROC-AUC**.


In [ ]:
# 6.2 Air Temperature Distribution
fig, ax = plt.subplots(figsize=(7, 5))
sns.histplot(df['Air temperature [K]'], kde=True, color='#1f77b4', ax=ax, bins=30, edgecolor='black')
ax.axvline(df['Air temperature [K]'].mean(), color='red', linestyle='--', linewidth=1.5, label=f"Mean: {df['Air temperature [K]'].mean():.2f} K")
ax.axvline(df['Air temperature [K]'].median(), color='green', linestyle=':', linewidth=1.5, label=f"Median: {df['Air temperature [K]'].median():.2f} K")
ax.set_title('Distribution of Ambient Air Temperature [K]', fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Air Temperature [K]', fontsize=11, fontweight='bold')
ax.set_ylabel('Frequency', fontsize=11, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()


### Graph 2 — Air Temperature Distribution
- **Purpose**: Examine the spread, operating range, and symmetry of ambient factory air temperatures.
- **Observation**: Ranges from 295.3 K to 304.5 K with Mean = 300.00 K and Median = 300.10 K. The distribution is symmetric and approximately Gaussian.
- **Meaning**: Factory floor ambient temperature fluctuates across seasonal ranges (~22°C to ~31°C) without artificial clamping.
- **ML Relevance**: Air temperature is well-conditioned for standard linear and distance-based estimators after standard z-score normalization.


In [ ]:
# 6.3 Process Temperature Distribution
fig, ax = plt.subplots(figsize=(7, 5))
sns.histplot(df['Process temperature [K]'], kde=True, color='#2ca02c', ax=ax, bins=30, edgecolor='black')
ax.axvline(df['Process temperature [K]'].mean(), color='red', linestyle='--', linewidth=1.5, label=f"Mean: {df['Process temperature [K]'].mean():.2f} K")
ax.axvline(df['Process temperature [K]'].median(), color='blue', linestyle=':', linewidth=1.5, label=f"Median: {df['Process temperature [K]'].median():.2f} K")
ax.set_title('Distribution of Internal Process Temperature [K]', fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Process Temperature [K]', fontsize=11, fontweight='bold')
ax.set_ylabel('Frequency', fontsize=11, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()


### Graph 3 — Process Temperature Distribution
- **Purpose**: Analyze the distribution of active cutting process temperatures.
- **Observation**: Ranges from 305.7 K to 313.8 K with Mean = 310.01 K and Median = 310.10 K, tracking approximately 10 K higher than ambient air.
- **Meaning**: Internal milling temperature is thermodynamically coupled with ambient conditions.
- **ML Relevance**: Because process temperature closely tracks air temperature, their absolute values exhibit strong collinearity. Engineering a `Temp_Difference` feature captures heat dissipation capacity directly.


In [ ]:
# 6.4 Rotational Speed Distribution
fig, ax = plt.subplots(figsize=(7, 5))
sns.histplot(df['Rotational speed [rpm]'], kde=True, color='#9467bd', ax=ax, bins=35, edgecolor='black')
ax.axvline(df['Rotational speed [rpm]'].mean(), color='red', linestyle='--', linewidth=1.5, label=f"Mean: {df['Rotational speed [rpm]'].mean():.1f} rpm")
ax.axvline(df['Rotational speed [rpm]'].median(), color='orange', linestyle=':', linewidth=1.5, label=f"Median: {df['Rotational speed [rpm]'].median():.1f} rpm")
ax.set_title('Distribution of Spindle Rotational Speed [rpm]', fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Rotational Speed [rpm]', fontsize=11, fontweight='bold')
ax.set_ylabel('Frequency', fontsize=11, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()


### Graph 4 — Rotational Speed Distribution
- **Purpose**: Determine the operational spindle speed profiles and examine right-tail skewness.
- **Observation**: Mean speed is 1,538.8 rpm, median is 1,503.0 rpm, and values extend up to 2,886 rpm with positive right-skewness.
- **Meaning**: Milling operations run predominantly between 1,400–1,600 rpm, but high-speed finishing operations introduce legitimate high-value operating points.
- **ML Relevance**: The 418 upper IQR observations represent valid extreme operating regimes rather than sensor transmission errors; they should not be dropped.


In [ ]:
# 6.5 Torque Distribution
fig, ax = plt.subplots(figsize=(7, 5))
sns.histplot(df['Torque [Nm]'], kde=True, color='#ff7f0e', ax=ax, bins=30, edgecolor='black')
ax.axvline(df['Torque [Nm]'].mean(), color='red', linestyle='--', linewidth=1.5, label=f"Mean: {df['Torque [Nm]'].mean():.2f} Nm")
ax.axvline(df['Torque [Nm]'].median(), color='blue', linestyle=':', linewidth=1.5, label=f"Median: {df['Torque [Nm]'].median():.2f} Nm")
ax.set_title('Distribution of Machine Torque [Nm]', fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Torque [Nm]', fontsize=11, fontweight='bold')
ax.set_ylabel('Frequency', fontsize=11, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()


### Graph 5 — Torque Distribution
- **Purpose**: Analyze load torque applied during milling operations.
- **Observation**: Normally distributed around Mean = 39.99 Nm and Median = 40.10 Nm (std = 9.97 Nm), with values spanning 3.8 Nm to 76.6 Nm.
- **Meaning**: Mechanical torque exhibits symmetric distribution, centered around 40 Nm.
- **ML Relevance**: Torque is an ideal feature for parametric models. Combining Torque with Rotational Speed yields cutting power, an informative physics-based feature.


In [ ]:
# 6.6 Tool Wear Distribution
fig, ax = plt.subplots(figsize=(7, 5))
sns.histplot(df['Tool wear [min]'], kde=True, color='#8c564b', ax=ax, bins=30, edgecolor='black')
ax.axvline(df['Tool wear [min]'].mean(), color='red', linestyle='--', linewidth=1.5, label=f"Mean: {df['Tool wear [min]'].mean():.1f} min")
ax.axvline(df['Tool wear [min]'].median(), color='green', linestyle=':', linewidth=1.5, label=f"Median: {df['Tool wear [min]'].median():.1f} min")
ax.set_title('Distribution of Cumulative Tool Wear [min]', fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Tool Wear [min]', fontsize=11, fontweight='bold')
ax.set_ylabel('Frequency', fontsize=11, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()


### Graph 6 — Tool Wear Distribution
- **Purpose**: Understand the lifespan distribution and replacement cycle of machine cutting tools.
- **Observation**: Uniformly distributed from 0 to ~210 minutes, with a tapering upper tail up to 253 minutes. Mean is 108.0 min, median is 108.0 min.
- **Meaning**: Tools are continuously used across manufacturing batches and periodically reset to 0 upon replacement.
- **ML Relevance**: Continuous uniform distribution confirms `Tool wear [min]` is an excellent target for regression tracking.


In [ ]:
# 6.7 Correlation Heatmap
fig, ax = plt.subplots(figsize=(8, 6.5))
corr_cols = ['Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]', 'Machine failure']
corr_mat = df[corr_cols].corr()
sns.heatmap(corr_mat, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1, linewidths=0.5, cbar_kws={'label': 'Pearson Correlation'}, ax=ax)
ax.set_title('Correlation Heatmap of Machine Operating Variables', fontsize=13, fontweight='bold', pad=12)
plt.tight_layout()
plt.show()


### Graph 7 — Correlation Heatmap
- **Purpose**: Quantify linear relationships, collinearity, and target correlations among continuous variables.
- **Observation**:
  - `Air temperature` and `Process temperature` have a strong positive correlation ($r = 0.88$).
  - `Rotational speed` and `Torque` have a strong negative correlation ($r = -0.88$), adhering to mechanical power laws ($P = \tau \cdot \omega$).
  - `Machine failure` correlates most positively with `Torque` ($r = 0.19$) and `Tool wear` ($r = 0.11$).
- **Meaning**: Strong physical laws govern the relationships between speed/torque and ambient/process temperatures.
- **ML Relevance**: Ridge/Lasso regularization will prevent coefficient variance explosion arising from multicollinear temperature and speed/torque pairs.


In [ ]:
# 6.8 Torque vs Machine Failure
fig, ax = plt.subplots(figsize=(7, 5))
sns.boxplot(x='Machine failure', y='Torque [Nm]', data=df, hue='Machine failure', palette=['#2b5c8f', '#d95f02'], ax=ax, width=0.4, legend=False)
ax.set_xticks([0, 1])
ax.set_xticklabels(['No Failure (0)', 'Failure (1)'])
ax.set_title('Operating Torque Distribution by Machine Failure Status', fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Machine Condition Status', fontsize=11, fontweight='bold')
ax.set_ylabel('Torque [Nm]', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()


### Graph 8 — Torque vs Machine Failure
- **Purpose**: Investigate whether machine failure observations display abnormal torque loads compared to healthy machines.
- **Observation**: Non-failing machines display a median torque of ~40.0 Nm. Machines suffering failure exhibit a significantly elevated median torque of ~53.3 Nm, with many failure points clustering in extreme bands (>60 Nm).
- **Meaning**: Excessive torque exerts severe shear stress on machine drive mechanisms, directly precipitating overstrain failure.
- **ML Relevance**: Torque is a primary discriminatory splitting feature for decision boundaries in classification.


In [ ]:
# 6.9 Tool Wear vs Machine Failure
fig, ax = plt.subplots(figsize=(7, 5))
sns.boxplot(x='Machine failure', y='Tool wear [min]', data=df, hue='Machine failure', palette=['#2b5c8f', '#d95f02'], ax=ax, width=0.4, legend=False)
ax.set_xticks([0, 1])
ax.set_xticklabels(['No Failure (0)', 'Failure (1)'])
ax.set_title('Tool Wear Accumulation by Machine Failure Status', fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Machine Condition Status', fontsize=11, fontweight='bold')
ax.set_ylabel('Tool Wear [min]', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()


## 7. Data Cleaning (Review 1 Rubric B1)
We perform rigorous data cleaning covering missing values, duplicates, and an IQR-based outlier audit.


In [ ]:
# 7.1 Missing Values Check
print("Total Missing Values in Dataset:", df.isnull().sum().sum())

# 7.2 Duplicates Check
print("Total Duplicate Rows in Dataset:", df.duplicated().sum())

# 7.3 Outlier Analysis via Interquartile Range (IQR)
print("
--- IQR Outlier Audit Across Numerical Sensor Variables ---")
num_cols = ['Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]']
outlier_summary = []

for col in num_cols:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    outliers = df[(df[col] < lower) | (df[col] > upper)]
    outlier_summary.append({
        'Feature': col,
        'Q1': round(q1, 2),
        'Q3': round(q3, 2),
        'IQR': round(iqr, 2),
        'Lower Bound': round(lower, 2),
        'Upper Bound': round(upper, 2),
        'Outlier Count': len(outliers),
        'Outlier %': round((len(outliers) / len(df)) * 100, 2)
    })

pd.DataFrame(outlier_summary)


### 7.4 Outlier Treatment Decision
- `Air temperature [K]` (0 outliers) & `Process temperature [K]` (0 outliers): No outliers detected.
- `Tool wear [min]` (0 outliers): Within normal operating life (0–253 min).
- `Torque [Nm]` (69 outliers, 0.69%): Extreme torque values (>67.2 Nm) coincide strongly with machine overstrain failures (`OSF`).
- `Rotational speed [rpm]` (418 outliers, 4.18%): High spindle speeds (>1,895 rpm) occur under light loads ($P = \tau \cdot \omega$) and correlate with power failures (`PWF`).
- **Engineering Decision**: **Retain all observations.** Industrial sensor values represent valid, critical operational dynamics. Truncating or dropping these records would eliminate the exact failure mechanisms our models are tasked to detect.
